# 05 · Busca de produtos

**Objetivo:** usar o sistema escolhido em `04_reranker.ipynb` para buscar qualquer texto no catalog: a hybrid search traz os 200 primeiros produtos e o reranker salvo em `models/` reordena os 100 primeiros. Toda a busca está em `wands_search.search.SearchPipeline`; este notebook só a chama.

**Regras deste notebook**
- **O test set não é usado.** Os exemplos com `label` (Etapa 2) usam queries do validation set.
- Nada é treinado aqui: o reranker é o modelo salvo por `uv run python -m wands_search.train_reranker`.

**Como ler este notebook**
- **Para buscar outro texto, troque `QUERY` na Etapa 1.1 e execute a célula.**
- Cada seção começa explicando a pergunta que ela responde e pode ser lida sozinha. As conclusões ficam logo abaixo da célula que as sustenta.
- Os termos técnicos ficam em inglês e estão definidos na seção "Termos usados no notebook"; são os mesmos de `04_reranker.ipynb`.

**Etapas**
1. Buscar qualquer texto
2. A lista antes e depois do reranker
3. Entradas inválidas
4. Latência

## TL;DR

- **Para buscar qualquer texto, troque `QUERY` na Etapa 1.1 e execute a célula.** A busca é o sistema escolhido no `04_reranker.ipynb`: hybrid search + reranker sem o cross-encoder. Na execução salva, ele carrega em 43 s, com os embeddings dos produtos lidos do cache, e o cross-encoder não é carregado (Setup).
- **Nas quatro buscas escritas por nós, os 5 primeiros resultados são do tipo pedido:** vanities de 36'', sofás cinza, cadeiras e conjuntos de mesa infantis e string lights (Etapa 1.2).
- **No validation set, o reranker melhora o NDCG@10 de 44 das 97 queries, piora 20 e não muda 33** (média +0,029). Na query de maior perda, "carpet 5x6", o top 10 passa a ter cinco tapetes de cadeira (chair mats) Irrelevant, que têm "carpet" no nome; antes, tinha um (Etapa 2).
- **Entradas inválidas:** texto vazio, só espaços ou que não é texto dão `InvalidQueryError`; só caracteres especiais dá 0 resultados; texto longo é cortado e avisado; `k` fora de 1 a 200 dá `ValueError` (Etapa 3).
- **Latência na execução salva:** mediana de 38,1 ms por busca (11,3 ms de retrieval e 26,7 ms de ranking), p95 de 45,6 ms, num CPU (Etapa 4).

## Termos usados no notebook

| Termo | Significado | Exemplo |
|---|---|---|
| query | Texto que a pessoa digita na busca. | "leather accent chair" |
| catalog | Os 42.994 produtos de `product.csv`. | |
| retrieval | Primeira etapa: escolhe, no catalog inteiro, os 200 primeiros produtos. Aqui, a **hybrid search**: RRF da BM25 v2 (lexical search) com a semantic search (embeddings de `BAAI/bge-small-en-v1.5`). | |
| candidate | Um dos 100 primeiros produtos do retrieval, os que o reranker reordena. | |
| reranker | Modelo LambdaMART (LightGBM) que dá uma nota a cada candidate a partir de 29 features (posição nos retrievals, cobertura das palavras da query em cada field, similaridade dos embeddings, features numéricas do produto) e reordena os 100 primeiros. As posições 101 a 200 seguem a ordem do retrieval. É o `hybrid_reranker_no_cross_encoder`, escolhido no `04_reranker.ipynb`, Etapa 7.3. | |
| score | A nota do reranker; só serve para ordenar os candidates de uma query, não para comparar queries. | |
| label | Julgamento humano de um par (query, produto): Exact, Partial ou Irrelevant. Produto **sem `label`**: nunca foi julgado para a query. | |
| NDCG@10 | Qualidade da ordem dos 10 primeiros, de 0 a 1, com gain Exact = 2, Partial = 1, Irrelevant e sem `label` = 0. | |
| latency_ms | Tempo de uma etapa da busca em milissegundos; p50 é a mediana e p95 o percentil 95. | |

## Setup

Carrega o catalog e monta o sistema a partir dos metadados do modelo salvo (`models/hybrid_reranker_no_cross_encoder.json`). O modelo não usa o cross-encoder, então ele não é carregado. Os embeddings dos produtos vêm de `cache/embeddings/`; sem cache, eles são gerados aqui, o que leva cerca de meia hora no CPU (ver o README).

In [1]:
import platform
import time

import numpy as np
import pandas as pd

from wands_search import data, evaluate, metrics, plots, ranking, reranker, retrieval, search

plots.apply_style()
pd.set_option("display.max_colwidth", 80)

products = data.load_products()
queries = data.load_queries()
labels = data.deduplicate_labels(data.load_labels())
labels_by_query = metrics.labels_by_query(labels)
query_split = evaluate.load_split()
validation_queries = evaluate.split_queries(queries, query_split, "validation")

start = time.perf_counter()
pipeline = search.SearchPipeline.load(products)
load_seconds = time.perf_counter() - start
_, model_metadata = reranker.load_model(reranker.SELECTED_MODEL)

print(f"search pipeline loaded in {load_seconds:.0f} s | machine: {platform.platform()} ({platform.processor()})")
print(f"model: {model_metadata['name']} | retrieval: {model_metadata['retrieval']} | features: {len(model_metadata['feature_columns'])} "
      f"| cross-encoder loaded: {pipeline.builder.cross_encoder is not None} | trained on {model_metadata['train_queries']} train queries")

search pipeline loaded in 43 s | machine: macOS-15.7.4-x86_64-i386-64bit-Mach-O (i386)
model: hybrid_reranker_no_cross_encoder | retrieval: hybrid | features: 29 | cross-encoder loaded: False | trained on 287 train queries


## Etapa 1 · Buscar qualquer texto

### 1.1 · Uma busca

Troque o texto de `QUERY` e execute a célula. `k` é o número de resultados mostrados, de 1 a 200.

In [2]:
QUERY = "leather accent chair"
response = pipeline.search(QUERY, k=10)
print(f"retrieval {response.retrieval_ms:.1f} ms | ranking {response.ranking_ms:.1f} ms | query cut: {response.truncated}")
plots.table(response.results.set_index("rank"), f"Top 10 products for the query '{QUERY}'", precision=3,
            note="score: reranker score, only meaningful to order the products of this query. reranked: the product was one of "
                 "the top 100 candidates that the reranker ordered.")

retrieval 15.3 ms | ranking 27.4 ms | query cut: False


,product_id,product_name,product_class,score,reranked
rank,,,,,
1,"34,391",dougherty 36.25 '' wide top grain leather lounge chair,Accent Chairs,1.972,True
2,360,jossi 36 '' wide top grain leather club chair,Accent Chairs,0.565,True
3,"21,446",calanthe 34 '' wide genuine leather club chair,Accent Chairs,0.469,True
4,"21,715",faux leather single sofa accent chair with ottoman foot rest for living room ( beige ),Accent Chairs,0.376,True
5,"34,299",cheng 32.67 '' wide top grain leather barrel chair,Accent Chairs,0.319,True
6,"24,970",caswell 33 '' w leather lounge chair with wood frame,Reception Seating Chairs|Accent Chairs,0.186,True
7,"8,388",denmark 25.5 '' wide full grain leather side chair,Reception Seating Chairs|Accent Chairs,-0.076,True
8,"18,093",40.5 '' wide genuine leather club chair,Accent Chairs,-0.122,True
9,"18,089",28.5 '' wide top grain leather club chair,Accent Chairs,-0.150,True


**Resultados**
- **Os 10 primeiros são poltronas de couro** com `product_class` Accent Chairs (três também com Reception Seating Chairs). O quarto é de couro sintético ("faux leather").
- Na execução salva, a busca levou 15,3 ms de retrieval e 27,4 ms de ranking.

### 1.2 · Outras buscas

Quatro textos escritos por nós, que não estão em `query.csv`: um com medida, um com cor e estilo, um com público e um com uso. Mostramos os 5 primeiros de cada.

In [3]:
EXAMPLE_QUERIES = ["36 inch bathroom vanity", "modern grey sofa", "kids desk chair", "outdoor string lights"]
example_results = pd.concat({query_text: pipeline.search(query_text, k=5).results.set_index("rank")[["product_name", "product_class"]]
                             for query_text in EXAMPLE_QUERIES}, names=["query", "rank"])
plots.table(example_results, "Top 5 products for four queries written by us (not in query.csv)", max_text_length=80)

**Resultados**
- **"36 inch bathroom vanity":** os cinco são vanities de 36''; dois são só a base ("base only").
- **"modern grey sofa":** dois loveseats cinza "mid-century modern", dois sectionals cinza e, em quinto, "vartavar modern furniture sets", sem cor no nome.
- **"kids desk chair":** cadeiras infantis de mesa e conjuntos de mesa e cadeira infantis.
- **"outdoor string lights":** os cinco são string lights; quatro dizem "outdoor" ou "solar".

**Limitações**
- Sem `label`, a leitura é nossa: quatro buscas, julgadas por uma pessoa.

## Etapa 2 · A lista antes e depois do reranker

Para ver o que o reranker muda, comparamos os 10 primeiros da hybrid search com os 10 primeiros depois do reranker, com o `label` de cada produto, em duas queries do validation set: a de **maior ganho** e a de **maior perda** de NDCG@10 com o reranker. Escolher os extremos mostra o que o reranker faz de melhor e de pior; não representa uma query típica (a média está no `04_reranker.ipynb`, Etapa 4.1).

In [4]:
reranker_gain = {}
validation_lists = {}
for query_id, query_text in zip(validation_queries["query_id"], validation_queries["query"]):
    before = pipeline.retriever.search(query_text, search.MAX_RESULTS)["product_id"].tolist()
    after = pipeline.search(query_text, k=search.MAX_RESULTS).results["product_id"].tolist()
    validation_lists[query_id] = (before, after)
    query_labels = labels_by_query.get(query_id, {})
    reranker_gain[query_id] = metrics.ndcg_at_k(after, query_labels) - metrics.ndcg_at_k(before, query_labels)
reranker_gain = pd.Series(reranker_gain)
print(f"validation queries: {len(reranker_gain)} | NDCG@10 up: {(reranker_gain > 0).sum()} | down: {(reranker_gain < 0).sum()} "
      f"| equal: {(reranker_gain == 0).sum()} | mean change: {reranker_gain.mean():+.3f}")

query_text_of = validation_queries.set_index("query_id")["query"]
product_names = products.set_index("product_id")["product_name"]
for title, query_id in (("largest gain", reranker_gain.idxmax()), ("largest loss", reranker_gain.idxmin())):
    before, after = validation_lists[query_id]
    query_labels = labels_by_query.get(query_id, {})
    side_by_side = pd.DataFrame({
        "hybrid search": [product_names[p] for p in before[:10]],
        "label (hybrid)": [query_labels.get(p, ranking.NO_LABEL) for p in before[:10]],
        "after the reranker": [product_names[p] for p in after[:10]],
        "label (reranker)": [query_labels.get(p, ranking.NO_LABEL) for p in after[:10]],
    }, index=pd.RangeIndex(1, 11, name="rank"))
    display(plots.table(side_by_side, f"Validation query '{query_text_of[query_id]}' ({title}): top 10 before and after the reranker",
                        max_text_length=60, note=f"NDCG@10: {metrics.ndcg_at_k(before, query_labels):.3f} before, "
                                                 f"{metrics.ndcg_at_k(after, query_labels):.3f} after. no label: never judged for this query."))

validation queries: 97 | NDCG@10 up: 44 | down: 20 | equal: 33 | mean change: +0.029


,hybrid search,label (hybrid),after the reranker,label (reranker)
rank,,,,
1,sande 12 '' x 24 '' porcelain field tile,no label,academy porcelain random mosaic wall & floor tile,Exact
2,sandstone 12 '' x 24 '' porcelain singular tile,no label,ronbo 6 '' x 10 '' porcelain wall & floor tile,Exact
3,ronbo 6 '' x 10 '' porcelain wall & floor tile,Exact,rocky 12 '' x 24 '' porcelain marble look wall & floor tile,Partial
4,rocky 12 '' x 24 '' porcelain marble look wall & floor tile,Partial,18 '' x 18 '' patterned wall & floor tile,Partial
5,rama 7 '' x 16 '' porcelain wood look wall & floor tile,Partial,rama 7 '' x 16 '' porcelain wood look wall & floor tile,Partial
6,academy porcelain random mosaic wall & floor tile,Exact,ronbo 6 '' x 10 '' porcelain patterned floor use tile,Partial
7,facade 2 '' x 2 '' porcelain mosaic tile,no label,facade 2 '' x 2 '' porcelain mosaic tile,no label
8,18 '' x 18 '' patterned wall & floor tile,Partial,allora 9 '' x 10 '' porcelain patterned wall & floor tile,Exact
9,allora 9 '' x 10 '' porcelain patterned wall & floor tile,Exact,academy random sized straight edge porcelain versailles m...,Exact


,hybrid search,label (hybrid),after the reranker,label (reranker)
rank,,,,
1,19.7 '' x 39.4 '' multi level loop carpet tile,Partial,12 '' x 12 '' level loop carpet tile,Partial
2,23.6 '' x 23.6 '' multi level loop carpet tile,Partial,19.7 '' x 39.4 '' multi level loop carpet tile,Partial
3,12 '' x 12 '' level loop carpet tile,Partial,9 '' x 36 '' level loop carpet tile,Partial
4,9 '' x 36 '' level loop carpet tile,Partial,supermat medium pile carpet beveled edge chair mat,Irrelevant
5,krol hand knotted wool brick red rug,Partial,low pile carpet straight edge chair mat,Irrelevant
6,nahri task chair,no label,high pile carpet straight edge chair mat,Irrelevant
7,multi area rug,no label,medium pile carpet beveled rectangular edge chair mat,Irrelevant
8,munroe tribal botemir oriental hand-knotted wool cream/br...,Partial,23.6 '' x 23.6 '' multi level loop carpet tile,Partial
9,supermat medium pile carpet beveled edge chair mat,Irrelevant,polycarbonate medium pile carpet straight edge chair mat,Irrelevant


**Resultados**
- **O reranker melhora o NDCG@10 de 44 das 97 queries do validation set, piora 20 e não muda 33;** a média sobe +0,029.
- **Maior ganho, "non slip shower floor tile" (NDCG@10 de 0,411 para 0,711):** a hybrid search tem dois azulejos sem `label` nas posições 1 e 2; o reranker põe dois Exact no lugar deles.
- **Maior perda, "carpet 5x6" (de 0,782 para 0,538):** o top 10 passa a ter cinco tapetes de cadeira (chair mats) Irrelevant (antes, um) e perde três tapetes Partial ("krol hand knotted wool brick red rug", "munroe tribal botemir oriental hand-knotted wool" e "mohawk green/beige area rug").

**O que isso implica (nossa interpretação)**
- Nos dois extremos, o reranker se apoia nas palavras da query presentes no `product_name` (`coverage_product_name` é a segunda feature mais importante no `04_reranker.ipynb`, Etapa 5.4). Isso ajuda quando o nome descreve o produto e atrapalha quando a palavra da query aparece num produto de outro tipo: "carpet" está no nome dos chair mats ("medium pile carpet ... chair mat").
- Em "carpet 5x6", a medida 5x6 vira um único token, `5x6`, que quase nenhum produto tem no texto, então `number_match` não ajuda.

**Limitações**
- Duas queries, escolhidas pelos extremos; não representam uma query típica.

## Etapa 3 · Entradas inválidas

A busca valida a entrada antes de qualquer trabalho (`retrieval.prepare_query`), com as mesmas regras do `02_bm25.ipynb`, Etapa 2:
- query que não é texto, vazia ou só com espaços: erro `InvalidQueryError`;
- query só com caracteres especiais, sem nenhum token: válida, sem resultados;
- query com mais de 500 caracteres ou 32 tokens: cortada, com `truncated = True`;
- `k` fora de 1 a 200, ou que não é inteiro: erro `ValueError`.

In [5]:
INVALID_INPUTS = {
    "empty text": ("", 10),
    "only spaces": ("   ", 10),
    "None": (None, 10),
    "a number": (42, 10),
    "only special characters": ("&/()!?", 10),
    "word not in the catalog": ("minnestrista", 10),
    "very long text (2,000 characters)": ("leather chair " * 143, 10),
    "accents and capitals": ("CADEIRA Écru Leather", 10),
    "k = 0": ("leather chair", 0),
    "k = 500": ("leather chair", 500),
}
outcomes = {}
for case, (query_value, k) in INVALID_INPUTS.items():
    try:
        response = pipeline.search(query_value, k=k)
        outcomes[case] = {"outcome": f"{len(response.results)} results", "query cut": response.truncated,
                          "first result": response.results["product_name"].iloc[0] if len(response.results) else "–"}
    except (retrieval.InvalidQueryError, ValueError) as error:
        outcomes[case] = {"outcome": f"{type(error).__name__}: {error}", "query cut": "–", "first result": "–"}
plots.table(pd.DataFrame(outcomes).T.rename_axis("input"), "What the search returns for invalid or unusual inputs (k = 10 unless stated)",
            max_text_length=70)

,outcome,query cut,first result
input,,,
empty text,InvalidQueryError: query is empty,–,–
only spaces,InvalidQueryError: query is empty,–,–
None,"InvalidQueryError: query must be a string, got NoneType",–,–
a number,"InvalidQueryError: query must be a string, got int",–,–
only special characters,0 results,False,–
word not in the catalog,10 results,False,minimania 1 light wall sconce
"very long text (2,000 characters)",10 results,True,vanwyk 23.62 '' w leather seat waiting room chair with metal frame
accents and capitals,10 results,False,devito 64 '' leather match tuxedo arm loveseat
k = 0,"ValueError: k must be an integer from 1 to 200, got 0",–,–


**Resultados**
- **Texto vazio, só espaços, `None` e um número dão `InvalidQueryError`,** com a mensagem do motivo, antes de qualquer busca.
- **Só caracteres especiais (`&/()!?`) é válido e dá 0 resultados.**
- **Uma palavra que o catalog não tem ("minnestrista") dá 10 resultados:** a semantic search sempre encontra vizinhos, mesmo sem palavras em comum. O primeiro é "minimania 1 light wall sconce".
- **Um texto de 2.000 caracteres é cortado em 500 e avisado** (`query cut = True`), e a busca continua.
- **Acentos e maiúsculas não quebram a busca:** "CADEIRA Écru Leather" dá 10 resultados; o primeiro é um loveseat de couro.
- **`k = 0` e `k = 500` dão `ValueError`.**

## Etapa 4 · Latência

Tempo de cada etapa da busca nas 97 queries do validation set, com `k = 10`, depois de 20 buscas de aquecimento. Os números mudam a cada execução e dependem da máquina (CPU, sem GPU).

In [6]:
for query_text in validation_queries["query"].head(evaluate.WARMUP_QUERIES):
    pipeline.search(query_text)
timings = pd.DataFrame([
    {"retrieval": response.retrieval_ms, "ranking": response.ranking_ms, "total": response.retrieval_ms + response.ranking_ms}
    for response in (pipeline.search(query_text) for query_text in validation_queries["query"])
])
plots.table(timings.quantile([0.5, 0.95]).set_axis(["p50", "p95"]).T.rename_axis("step"),
            "Latency of one search in milliseconds (hybrid search + reranker without the cross-encoder), 97 validation queries",
            precision=1, note=f"One process on {platform.platform()} ({platform.processor()}), CPU only, after {evaluate.WARMUP_QUERIES} "
                              "warm-up searches. Retrieval: top 200 of the hybrid search. Ranking: features of the top 100 and the "
                              "reranker score.")

,p50,p95
step,,
retrieval,11.3,14.8
ranking,26.7,31.1
total,38.1,45.6


**Resultados**
- **Na execução salva, uma busca leva 38,1 ms na mediana e 45,6 ms no p95:** 11,3 ms (p50) de retrieval e 26,7 ms de ranking.
- O ranking inclui o embedding da query, calculado de novo para as features de embedding.

**Limitações**
- Uma única execução numa única máquina, sem GPU.